# Import Modules

In [114]:
import sys
from pathlib import Path

# Add project root to path
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Our modules
from src.data.connector_factory import ConnectorFactory
from src.models.bsm import implied_volatility
from src.models.svi import fit_svi_smile, svi_total_variance, svi_iv
from src.models.ssvi import calibrate_ssvi, ssvi_iv
from src.utils.time_utils import compute_time_to_expiry
from src.utils.data_preprocessing import filter_otm_for_calibration

# For reproducability
np.random.seed(42)

# Connector & Parameters

In [100]:
SYMBOL = "SPY"
MAX_EXPIRIES = 5  # Use the nearest 4 expiries for a clean surface
R = 0.045  # Risk-free rate (adjust to current market)
Q = 0.012  # Dividend yield for SPY

# Connect to data source
connector = ConnectorFactory.get_connector(provider="yfinance", symbol=SYMBOL)
spot = connector.get_spot_price()

print(f"Symbol: {SYMBOL}")
print(f"Spot Price: {spot:.2f}")
print(f"r: {R:.3f}, q: {Q:.3f}")

Symbol: SPY
Spot Price: 747.71
r: 0.045, q: 0.012


# Get options data

In [101]:
expiries = connector.get_available_expiries()[1:MAX_EXPIRIES]
print(f"Fetched {len(expiries)} expiries: {expiries}")

raw_chains = {}
for exp in expiries:
    raw_chains[exp] = connector.get_chain_for_expiry(exp)
    T = compute_time_to_expiry(exp)
    print(f"  {exp}: T={T:.4f} years, Calls={len(raw_chains[exp]['calls'])}, Puts={len(raw_chains[exp]['puts'])}")

Fetched 4 expiries: ('2026-07-08', '2026-07-09', '2026-07-10', '2026-07-13')
  2026-07-08: T=0.0025 years, Calls=54, Puts=63
  2026-07-09: T=0.0052 years, Calls=56, Puts=67
  2026-07-10: T=0.0080 years, Calls=141, Puts=135
  2026-07-13: T=0.0162 years, Calls=51, Puts=73


# Calibrate SVI slices to each option chain (expiry)

In [115]:
def prepare_svi_slice(expiry, raw_data):
    """Prepare a single expiry slice for SVI calibration."""
    T = compute_time_to_expiry(expiry)
    forward = spot * np.exp((R - Q) * T)

    # Compute IVs for ALL options first
    all_ops = raw_data["calls"] + raw_data["puts"]
    for opt in all_ops:
        if opt.mid > 0 and opt.bid > 0 and opt.ask > 0:
            iv = implied_volatility(opt.mid, spot, opt.strike, T, R, Q, opt.option_type)
            if not np.isnan(iv):
                opt.implied_vol = iv

    # Filter OTM (using our utility)
    strikes, ivs = filter_otm_for_calibration(all_ops, spot, T, R, Q, max_spread_pct=0.5)

    if len(strikes) < 5:
        print(f"⚠️  {expiry}: Only {len(strikes)} OTM options found. Skipping.")
        return None

    # Calibrate Raw SVI to get a clean theta (a in SVI = ATM total variance)
    result = fit_svi_smile(strikes, ivs, T, forward, r=R, q=Q)

    if not result.success:
        print(f"❌ {expiry}: SVI calibration failed. Error message: {result.message}")
        return None

    a, b, rho, m, sigma = result.params
    theta = svi_total_variance(0, a, b, rho, m, sigma) # ATM variance is equal to the total variance at k=0

    # Compute log-moneyness for SSVI input
    k = np.log(strikes / forward)

    # Visualize fit
    k_grid = np.linspace(min(k), max(k), 50)
    iv_svi = svi_iv(k_grid, a, b, rho, m, sigma, T)
    strike_grid = np.exp(k_grid) * forward
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=strikes, y=ivs, mode='markers', name='Market IV'))
    fig.add_trace(go.Scatter(x=strike_grid, y=iv_svi, mode='lines', name='SVI Fit'))
    fig.show()

    # Return
    return {
        "expiry": expiry,
        "T": T,
        "forward": forward,
        "strikes": strikes,
        "ivs": ivs,
        "k": k,
        "theta": theta,
        "svi_params": result.params,
    }

# Prepare slices using helper function
svi_slices = []
for exp, raw in raw_chains.items():
    slice_data = prepare_svi_slice(exp, raw)
    if slice_data is not None:
        svi_slices.append(slice_data)

print(f"\n✅ Successfully prepared {len(svi_slices)} SVI slices.")



✅ Successfully prepared 4 SVI slices.


# Build the SSVI input dictionary

In [113]:
# The SSVI calibrator expects a dict: {expiry: {'k': array, 'iv': array, 'theta': float, 'T': float}}
ssvi_input = {}
for slice_data in svi_slices:
    exp = slice_data["expiry"]
    ssvi_input[exp] = {
        "k": slice_data["k"],
        "iv": slice_data["ivs"],
        "theta": slice_data["theta"],
        "T": slice_data["T"],
    }

print("SSVI Input Summary:")
for exp, data in ssvi_input.items():
    print(f"  {exp}: T={data['T']:.4f}, theta={data['theta']:.4f}, points={len(data['k'])}")


SSVI Input Summary:
  2026-07-08: T=0.0025, theta=0.0001, points=54
  2026-07-09: T=0.0052, theta=0.0001, points=67
  2026-07-10: T=0.0080, theta=0.0016, points=117
  2026-07-13: T=0.0162, theta=0.0002, points=85


# Run Global SSVI Calibration

In [130]:
gamma = 0.5 # Square-root SSVI
ssvi_result = calibrate_ssvi(ssvi_input, gamma=gamma)

if ssvi_result.success:
    rho, eta = ssvi_result.params
    print(f"\n✅ SSVI Calibration SUCCESSFUL!")
    print(f"   rho (correlation): {rho:.4f}")
    print(f"   eta (skew decay): {eta:.4f}")
    print(f"   gamma (fixed): {gamma:.1f}")
    print(f"   Message: {ssvi_result.message}")
else:
    print(f"\n❌ SSVI Calibration FAILED: {ssvi_result.message}")
    raise RuntimeError("Stop here.")


✅ SSVI Calibration SUCCESSFUL!
   rho (correlation): 0.4277
   eta (skew decay): 1.0322
   gamma (fixed): 0.5
   Message: Calibration successful. Iterations: 14


# Verify No-Arbitrage Constraints

In [131]:
print("\n🔍 No-Arbitrage Constraint Check:")
limit = 4.0 / (1.0 + abs(rho))
print(f"   Calendar constraint: eta ({eta:.4f}) <= {limit:.4f} ? {eta <= limit}")

print("\n   Butterfly constraints (theta * phi <= limit):")
for exp, data in ssvi_input.items():
    theta = data['theta']
    phi = eta / (theta ** gamma)
    val = theta * phi
    check = val <= limit
    print(f"     {exp}: theta*phi = {val:.4f} <= {limit:.4f} ? {check}")



🔍 No-Arbitrage Constraint Check:
   Calendar constraint: eta (1.0322) <= 2.8017 ? True

   Butterfly constraints (theta * phi <= limit):
     2026-07-08: theta*phi = 0.0078 <= 2.8017 ? True
     2026-07-09: theta*phi = 0.0097 <= 2.8017 ? True
     2026-07-10: theta*phi = 0.0417 <= 2.8017 ? True
     2026-07-13: theta*phi = 0.0142 <= 2.8017 ? True


# Visualization: Per-Expiry Fits (Market IV vs SSVI)

In [133]:
num_slices = len(svi_slices)
if num_slices == 0:
    raise ValueError("No slices to plot.")

fig = make_subplots(
    rows=1, cols=num_slices,
    subplot_titles=[f"{s['expiry']} (T={s['T']:.3f})" for s in svi_slices],
    shared_yaxes=True,
    horizontal_spacing=0.08
)

rho_fit, eta_fit = ssvi_result.params

for i, slice_data in enumerate(svi_slices):
    expiry = slice_data['expiry']
    strikes = slice_data['strikes']
    iv_market = slice_data['ivs']
    T = slice_data['T']
    theta = slice_data['theta']
    k = slice_data['k']

    # Compute SSVI fitted IVs
    iv_ssvi = ssvi_iv(k, theta, rho_fit, eta_fit, T, gamma)

    # Add market IVs
    fig.add_trace(
        go.Scatter(
            x=strikes,
            y=iv_market,
            mode='markers',
            name='Market IV' if i == 0 else '',
            marker=dict(color='blue', size=6),
            showlegend=(i == 0)
        ),
        row=1, col=i+1
    )

    # Add SSVI fit
    fig.add_trace(
        go.Scatter(
            x=strikes,
            y=iv_ssvi,
            mode='markers',
            name='SSVI Fit' if i == 0 else '',
            line=dict(color='red', width=2),
            showlegend=(i == 0)
        ),
        row=1, col=i+1
    )

    fig.update_xaxes(title_text="Strike", row=1, col=i+1)

fig.update_yaxes(title_text="Implied Volatility", row=1, col=1)
fig.update_layout(
    title_text=f"SSVI Fit vs Market IVs (rho={rho_fit:.3f}, eta={eta_fit:.3f})",
    height=500,
    width=1200,
    hovermode='x unified'
)
fig.show()

# Visualization: 3D Volatility Surface

In [134]:
# Create a dense grid for the 3D surface
strike_range = np.linspace(
    min([s['strikes'].min() for s in svi_slices]),
    max([s['strikes'].max() for s in svi_slices]),
    30
)

T_range = np.array([s['T'] for s in svi_slices])

IV_matrix = []
for T_val in T_range:
    # For each expiry, compute the SSVI IV for all strikes in the dense grid
    # We need to map strikes to log-moneyness using the forward for that expiry.
    # We'll approximate by finding the theta for that expiry.
    slice_data = next(s for s in svi_slices if abs(s['T'] - T_val) < 0.001)
    forward = slice_data['forward']
    theta = slice_data['theta']
    k_grid = np.log(strike_range / forward)
    iv_grid = ssvi_iv(k_grid, theta, rho_fit, eta_fit, T_val, gamma)
    IV_matrix.append(iv_grid)

IV_matrix = np.array(IV_matrix)

# 3D Surface Plot
fig3d = go.Figure(data=[
    go.Surface(
        z=IV_matrix,
        x=strike_range,
        y=T_range * 365,  # Convert to days for readability
        colorscale='Viridis',
        hovertemplate='Strike: %{x:.1f}<br>DTE: %{y:.0f}<br>IV: %{z:.2%}<extra></extra>'
    )
])

fig3d.update_layout(
    title_text=f"SSVI Volatility Surface (rho={rho_fit:.3f}, eta={eta_fit:.3f})",
    scene=dict(
        xaxis_title='Strike',
        yaxis_title='Days to Expiry',
        zaxis_title='Implied Volatility',
        camera=dict(eye=dict(x=1.5, y=1.5, z=1.2))
    ),
    height=600,
    width=900
)
fig3d.show()